# 🕵️ Casefile: Missing Person Location Prediction System
### A Beginner-Friendly, End-to-End Machine Learning Pipeline

Welcome! This notebook walks through a complete machine learning workflow, step by step, in separate cells so you can run, inspect, and tweak each part individually.

**What this notebook does:**
1. Loads and audits a dataset of missing person cases
2. Engineers useful features from raw columns
3. Splits data and builds simple baselines to compare against
4. Trains several classification models
5. Evaluates them with accuracy, Top-K accuracy, and F1 scores
6. Saves the best model and uses it to generate a sample prediction report

> ⚠️ **Ethical note:** This is a *statistical prioritization tool*, not a definitive prediction system. Outputs are probabilities meant to help prioritize search efforts — never treat them as certainties.


## 📦 Step 0: Import Libraries
All the libraries we need for data handling, preprocessing, modeling, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Suppress minor warnings for clean student output
warnings.filterwarnings('ignore')

print("=" * 65)
print(" CASEFILE: MISSING PERSON LOCATION PREDICTION SYSTEM")
print(" Student End-to-End Machine Learning Pipeline")
print("=" * 65)


## 📂 Step 1: Loading and Auditing the Dataset

We try to load a real CSV file (`synthetic_case_cleaned.csv`). If it isn't found, we generate a **synthetic sample dataset** of 5,000 fake missing-person cases so the notebook still runs end-to-end for learning purposes.

We also do a quick data audit: row/column counts, missing values, and duplicates.

In [ ]:
print("\n--- STEP 1: LOADING AND AUDITING DATASET ---")
try:
    df = pd.read_csv('synthetic_case_cleaned.csv')
    print("Dataset successfully loaded from 'synthetic_case_cleaned.csv'!")
except FileNotFoundError:
    print("Warning: CSV file not found. Generating sample data for demonstration...")
    np.random.seed(42)
    n = 5000
    df = pd.DataFrame({
        'Case_ID': [f'CS-{i:04d}' for i in range(n)],
        'Person_ID': [f'P-{np.random.randint(1000, 2000)}' for i in range(n)],
        'Age_Group': np.random.choice(['Youth', 'Adult', 'Senior'], n),
        'Gender': np.random.choice(['Male', 'Female', 'Other'], n),
        'Last_Latitude': np.random.uniform(33.5, 34.5, n),
        'Last_Longitude': np.random.uniform(-118.5, -117.5, n),
        'Last_Seen_Time': pd.date_range(start='2026-01-01', periods=n, freq='H'),
        'Day': np.random.choice(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], n),
        'Weather': np.random.choice(['Clear', 'Rain', 'Fog', 'Snow'], n),
        'Usual_Area': np.random.choice([f'Area_{chr(65+i)}' for i in range(10)], n),
        'Average_Distance': np.random.exponential(5.0, n),
        'Average_Speed': np.random.exponential(2.5, n),
        'Previous_Area': np.random.choice([f'Area_{chr(65+i)}' for i in range(10)], n),
        'Time_Since_Last_Seen': np.random.uniform(1.0, 48.0, n),
        'Target_Area': np.random.choice([f'Area_{chr(65+i)}' for i in range(10)], n, p=[0.095, 0.147, 0.097, 0.12, 0.173, 0.111, 0.068, 0.113, 0.059, 0.016])
    })

print(f"Total Rows (Missing Person Cases): {df.shape[0]}")
print(f"Total Columns (Features & Target): {df.shape[1]}")
print(f"Missing Values Count: {df.isnull().sum().sum()}")
print(f"Duplicate Rows Count: {df.duplicated().sum()}")


Let's take a quick peek at the first few rows of the dataset:

In [ ]:
df.head()


## 🛠️ Step 2: Feature Engineering

Raw columns aren't always in a shape models can use well. Here we:
- Convert the `Last_Seen_Time` timestamp into an hour value
- Encode that hour **cyclically** (using sine/cosine) so midnight and 1 AM are recognized as close together, not far apart
- Compute movement-related ratios (`Speed_Distance_Ratio`, `Movement_Intensity`)
- Create a flag for whether the person's usual area matches their previous area

This is done inside a reusable function so we can apply the same transformation later to new, unseen cases at prediction time.

In [ ]:
def engineer_features(input_df):
    """
    Transforms raw columns into meaningful predictors without leaking target info.
    Converts timestamps into cyclical hour coordinates and computes movement ratios.
    """
    df_clean = input_df.copy()
    
    # Convert string timestamp to datetime and extract hour
    df_clean['Last_Seen_Time'] = pd.to_datetime(df_clean['Last_Seen_Time'], errors='coerce')
    df_clean['Last_Seen_Hour'] = df_clean['Last_Seen_Time'].dt.hour.fillna(12)
    
    # Cyclical hour encoding (handles midnight wrap-around mathematically)
    df_clean['Hour_Sin'] = np.sin(2 * np.pi * df_clean['Last_Seen_Hour'] / 24.0)
    df_clean['Hour_Cos'] = np.cos(2 * np.pi * df_clean['Last_Seen_Hour'] / 24.0)
    
    # Movement & spatial interactions
    df_clean['Speed_Distance_Ratio'] = df_clean['Average_Speed'] / (df_clean['Average_Distance'] + 1e-5)
    df_clean['Movement_Intensity'] = df_clean['Average_Speed'] * df_clean['Time_Since_Last_Seen']
    df_clean['Is_Usual_Prev_Match'] = (df_clean['Usual_Area'] == df_clean['Previous_Area']).astype(int)
    
    return df_clean


In [ ]:
df_engineered = engineer_features(df)
print("Feature engineering successfully completed!")
df_engineered.head()


## ✂️ Step 3: Train/Test Split & Baseline Evaluation

Before training any "real" model, it's good practice to build **simple baselines**. If our machine learning models can't beat these, they're not worth the added complexity!

- **Baseline 1 — Majority Class:** always predict the most common `Target_Area`
- **Baseline 2 — Usual Area Match:** predict the person's usual area
- **Baseline 3 — Previous Area Match:** predict the person's previous area

We also split the data 80/20 into training and testing sets, using **stratification** so each area is proportionally represented in both sets.

In [ ]:
print("\n--- STEP 3: TRAIN/TEST SPLIT & BASELINE EVALUATION ---")

# Drop IDs and raw timestamps from feature matrix
drop_cols = ['Case_ID', 'Person_ID', 'Last_Seen_Time', 'Target_Area']
feature_cols = [c for c in df_engineered.columns if c not in drop_cols]

X = df_engineered[feature_cols]
y = df_engineered['Target_Area']

# Encode target labels into integers (0 to 9)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 80% Training set, 20% Testing set with stratification for imbalanced classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)


In [ ]:
# Baseline 1: Majority Class Classifier
majority_class = np.bincount(y_train).argmax()
b1_preds = np.full_like(y_test, majority_class)
b1_acc = accuracy_score(y_test, b1_preds)

# Baseline 2 & 3: Rule-based mapping helpers
unique_areas = label_encoder.classes_
def evaluate_rule_baseline(col_name):
    preds = []
    col_idx = feature_cols.index(col_name)
    for row in X_test.values:
        val = row[col_idx]
        if val in unique_areas:
            preds.append(label_encoder.transform([val])[0])
        else:
            preds.append(majority_class)
    return accuracy_score(y_test, preds)

b2_acc = evaluate_rule_baseline('Usual_Area')
b3_acc = evaluate_rule_baseline('Previous_Area')

print(f"Baseline 1 (Majority Class):    {b1_acc*100:.2f}%")
print(f"Baseline 2 (Usual Area Match):  {b2_acc*100:.2f}%")
print(f"Baseline 3 (Prev Area Match):   {b3_acc*100:.2f}%")


## 🤖 Step 4: Model Training Pipeline

Now for the real models! We build a `ColumnTransformer` that:
- Scales numeric columns with `StandardScaler`
- One-hot encodes categorical columns with `OneHotEncoder`

Then we train **four different classifiers** inside their own `Pipeline` (so preprocessing + model are bundled together):
1. Logistic Regression
2. Random Forest
3. Gradient Boosting
4. XGBoost

In [ ]:
cat_cols = ['Age_Group', 'Gender', 'Day', 'Weather', 'Usual_Area', 'Previous_Area']
num_cols = [c for c in feature_cols if c not in cat_cols]

# Column transformer scales numeric columns and one-hot encodes categorical ones
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42, eval_metric='mlogloss')
}


In [ ]:
print("\n--- STEP 4: MODEL TRAINING PIPELINE ---")

trained_pipelines = {}
for name, model in models.items():
    print(f"Training {name} pipeline...")
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe

print("All models successfully trained!")


## 📊 Step 5: Model Evaluation & Top-K Metrics

Since this is a **prioritization** tool rather than a single hard prediction, it's useful to check not just "did the model get the #1 area right?" but also "was the correct area in the model's Top-3 or Top-5 guesses?"

We compute:
- **Top-1, Top-3, Top-5 accuracy**
- **Macro F1** (treats every class equally, good for imbalance)
- **Weighted F1** (accounts for class frequency)

In [ ]:
def evaluate_top_k(proba_matrix, y_true, k_vals=[1, 3, 5]):
    """Calculates Top-1, Top-3, and Top-5 prediction accuracy."""
    results = {}
    sorted_preds = np.argsort(proba_matrix, axis=1)[:, ::-1]
    for k in k_vals:
        correct = sum(1 for i, true_label in enumerate(y_true) if true_label in sorted_preds[i, :k])
        results[k] = correct / len(y_true)
    return results


In [ ]:
print("\n--- STEP 5: MODEL EVALUATION & TOP-K METRICS ---")

summary_results = [
    {'Model': 'Majority Baseline', 'Top-1 Accuracy': b1_acc, 'Top-3': b1_acc, 'Top-5': b1_acc, 'Macro F1': 0.0, 'Weighted F1': 0.0},
    {'Model': 'Usual Area Baseline', 'Top-1 Accuracy': b2_acc, 'Top-3': b2_acc, 'Top-5': b2_acc, 'Macro F1': 0.0, 'Weighted F1': 0.0},
    {'Model': 'Previous Area Baseline', 'Top-1 Accuracy': b3_acc, 'Top-3': b3_acc, 'Top-5': b3_acc, 'Macro F1': 0.0, 'Weighted F1': 0.0}
]

for name, pipe in trained_pipelines.items():
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)
    top_k = evaluate_top_k(probs, y_test, [1, 3, 5])
    
    acc = accuracy_score(y_test, preds)
    _, _, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
    wf1 = precision_recall_fscore_support(y_test, preds, average='weighted', zero_division=0)[2]
    
    summary_results.append({
        'Model': name,
        'Top-1 Accuracy': acc,
        'Top-3': top_k[3],
        'Top-5': top_k[5],
        'Macro F1': f1,
        'Weighted F1': wf1
    })

df_results = pd.DataFrame(summary_results)
print("\n" + "=" * 80)
print(df_results.to_string(index=False))
print("=" * 80)


Let's also view the results as a nicely formatted DataFrame:

In [ ]:
df_results


## 💾 Step 6: Model Saving & Inference Function

Finally, we:
1. Save our best-performing model (**XGBoost**) and the label encoder to disk with `joblib`, so they can be reloaded later without retraining
2. Define a `predict_probable_areas()` function that takes a **single new case** and returns a Top-5 probability report, written with careful ethical phrasing (probabilities, not certainties)
3. Test it on a sample case

In [ ]:
print("\n--- STEP 6: MODEL SAVING & INFERENCE FUNCTION ---")

# Save our primary advanced model (XGBoost)
best_model = trained_pipelines['XGBoost']
joblib.dump(best_model, 'location_model.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
print("Model and label encoder successfully saved to disk using joblib!")


In [ ]:
def predict_probable_areas(case_input_dict):
    """
    Generates Top-5 probable location areas with strict ethical phrasing
    per project guidelines (stating probabilities, not guarantees).
    """
    input_df = pd.DataFrame([case_input_dict])
    processed_input = engineer_features(input_df)
    
    drop_cols = ['Case_ID', 'Person_ID', 'Last_Seen_Time', 'Target_Area']
    feature_cols = [c for c in processed_input.columns if c not in drop_cols]
    
    probs = best_model.predict_proba(processed_input[feature_cols])[0]
    top_indices = np.argsort(probs)[::-1]
    
    print("\n" + "=" * 55)
    print(" CASEFILE PROBABLE LOCATION PREDICTION REPORT")
    print("=" * 55)
    
    rank_labels = ['Very High', 'High', 'Medium', 'Low', 'Very Low']
    for rank in range(1, 6):
        idx = top_indices[rank - 1]
        area_name = label_encoder.inverse_transform([idx])[0]
        prob_val = probs[idx] * 100
        priority = rank_labels[rank - 1]
        print(f"{rank}. {area_name:<8} — {prob_val:5.1f}% — {priority} Priority")
        
    print("-" * 55)
    print("ETHICAL NOTICE: Model prediction is a statistical probability")
    print("estimate for search prioritization, not definitive proof.")
    print("=" * 55)


In [ ]:
df_results


### 🔎 Try it out with a sample case

In [ ]:
# Test the prediction function with a sample case record
sample_case = {
    'Case_ID': 'CS-9999',
    'Person_ID': 'P-5001',
    'Age_Group': 'Adult',
    'Gender': 'Female',
    'Last_Latitude': 34.0522,
    'Last_Longitude': -118.2437,
    'Last_Seen_Time': '2026-06-01 18:30:00',
    'Day': 'Friday',
    'Weather': 'Clear',
    'Usual_Area': 'Area_E',
    'Average_Distance': 4.5,
    'Average_Speed': 2.1,
    'Previous_Area': 'Area_B',
    'Time_Since_Last_Seen': 12.0
}

predict_probable_areas(sample_case)


---
## ✅ Recap

- We audited and engineered features from raw case data
- Built three simple baselines to set a performance floor
- Trained four classifiers (Logistic Regression, Random Forest, Gradient Boosting, XGBoost)
- Evaluated them with Top-K accuracy and F1 scores
- Saved the best model and generated a sample Top-5 prediction report

Feel free to experiment: try new features in Step 2, tune hyperparameters in Step 4, or plug in your own case in Step 6!